# Perplexity Prompt Tester

Iterates on the `_build_prompt` logic and tests it against specific leads.

**Workflow:**
1. Define leads inline (or load from task state)
2. Inspect what the current prompt generates
3. Edit `build_prompt_v2` to test improvements
4. Run Perplexity and compare raw responses side-by-side

## Cell 1: Setup

In [1]:
import sys, json, logging
sys.path.append("../..")

from email_finder.models import LeadInput
from email_finder.config import Config
from email_finder.discovery.perplexity_search import _build_prompt, parse_perplexity_response

logging.basicConfig(level=logging.INFO)
config = Config()
print("Config loaded.")

Config loaded.


## Cell 2: Define Test Leads

Load from saved task state, or define inline below.

In [2]:
# --- Option A: load failing leads from saved task state ---
STATE_PATH = "./output/mailin_task_state.json"

with open(STATE_PATH) as f:
    state = json.load(f)

all_leads = [LeadInput(**d) for d in state["leads"]]

# Focus on the leads that had no email found
FAILING = {"A Mommy And A Mic", "Paper Trails", "TravelRight.Today"}
test_leads = [l for l in all_leads if l.full_name in FAILING]

for l in test_leads:
    print(f"  {l.full_name:<35}  website={l.website}  domain={l.company_domain}")

  A Mommy And A Mic                    website=http://www.amommyandamic.com/  domain=None
  TravelRight.Today                    website=http://www.travelright.today/  domain=None
  Paper Trails                         website=https://papertrails.podbean.com/  domain=None


## Cell 3: Compare Prompts — Current vs New

In [3]:
def build_prompt_v2(lead: LeadInput) -> str:
    """
    Improved prompt that includes the known website/domain in the context
    so Perplexity can search/scrape the actual site's contact page.
    """
    ask_for = ["professional email address"]

    if not lead.linkedin_url:
        ask_for.append("LinkedIn profile URL")
    if not lead.company_domain and not lead.website:
        ask_for.append("company website or domain")
    if not lead.facebook_url:
        ask_for.append("Facebook page")
    if not lead.youtube_url:
        ask_for.append("YouTube channel")
    if not lead.twitter_url:
        ask_for.append("Twitter/X profile")

    context_parts = [lead.full_name]
    if lead.company_name and lead.company_name != lead.full_name:
        context_parts.append(f"at {lead.company_name}")
    if lead.podcast_name and lead.podcast_name != lead.full_name:
        context_parts.append(f'host/guest of "{lead.podcast_name}" podcast')
    if lead.website:
        context_parts.append(f"(website: {lead.website})")   # ← NEW: include website
    elif lead.company_domain:
        context_parts.append(f"(domain: {lead.company_domain})")  # ← NEW: fallback to domain
    if lead.linkedin_url:
        context_parts.append(f"LinkedIn: {lead.linkedin_url}")

    return (
        f"Find the {', '.join(ask_for)} for {' '.join(context_parts)}. "
        "Check the website's contact and about pages. "
        "Return only verified, factual information with sources."
    )


print(f"{'Lead':<35}  CURRENT PROMPT")
print("-" * 100)
for lead in test_leads:
    print(f"{lead.full_name:<35}  {_build_prompt(lead)}")

print()
print(f"{'Lead':<35}  V2 PROMPT")
print("-" * 100)
for lead in test_leads:
    print(f"{lead.full_name:<35}  {build_prompt_v2(lead)}")

Lead                                 CURRENT PROMPT
----------------------------------------------------------------------------------------------------
A Mommy And A Mic                    Find the professional email address, LinkedIn profile URL, Facebook page, YouTube channel, Twitter/X profile for A Mommy And A Mic. Return only verified, factual information with sources.
TravelRight.Today                    Find the professional email address, LinkedIn profile URL, Facebook page, YouTube channel, Twitter/X profile for TravelRight.Today. Return only verified, factual information with sources.
Paper Trails                         Find the professional email address, LinkedIn profile URL, Facebook page, YouTube channel, Twitter/X profile for Paper Trails. Return only verified, factual information with sources.

Lead                                 V2 PROMPT
----------------------------------------------------------------------------------------------------
A Mommy And A Mic           

## Cell 4: Run Perplexity with V2 Prompt

Set `PROMPT_FN` to switch between `_build_prompt` (current) and `build_prompt_v2`.

In [4]:
from llm_utils.gpt_utils import PerplexityService

PROMPT_FN = build_prompt_v2   # ← swap to _build_prompt to compare

_PERPLEXITY_MODEL = "sonar"
_TEMPERATURE      = 0.1
_MAX_TOKENS       = 500

service = PerplexityService(api_key=config.perplexity_api_key)

perplexity_results = {}   # lead_name → {prompt, raw_response, parsed}

for lead in test_leads:
    prompt = PROMPT_FN(lead)
    print(f"\n[{lead.full_name}]")
    print(f"  Prompt: {prompt}")

    response = service.process_content(
        content="",
        prompt=prompt,
        model=_PERPLEXITY_MODEL,
        temperature=_TEMPERATURE,
        max_tokens=_MAX_TOKENS,
    )

    if not response.get("success"):
        print(f"  ERROR: {response.get('error')}")
        continue

    raw = response.get("result", "")
    parsed = parse_perplexity_response(raw)

    perplexity_results[lead.full_name] = {
        "prompt": prompt,
        "raw_response": raw,
        "parsed": parsed,
    }

    print(f"  Emails found:  {parsed['emails']}")
    print(f"  Website found: {parsed['website']}")
    print(f"  LinkedIn:      {parsed['linkedin_url']}")


[A Mommy And A Mic]
  Prompt: Find the professional email address, LinkedIn profile URL, Facebook page, YouTube channel, Twitter/X profile for A Mommy And A Mic (website: http://www.amommyandamic.com/). Check the website's contact and about pages. Return only verified, factual information with sources.


INFO:httpx:HTTP Request: POST https://api.perplexity.ai/chat/completions "HTTP/1.1 200 OK"


  Emails found:  ['briggz@comcast.net']
  Website found: http://www.amommyandamic.com
  LinkedIn:      None

[TravelRight.Today]
  Prompt: Find the professional email address, LinkedIn profile URL, Facebook page, YouTube channel, Twitter/X profile for TravelRight.Today (website: http://www.travelright.today/). Check the website's contact and about pages. Return only verified, factual information with sources.


INFO:httpx:HTTP Request: POST https://api.perplexity.ai/chat/completions "HTTP/1.1 200 OK"


  Emails found:  ['info@travelright.today']
  Website found: None
  LinkedIn:      None

[Paper Trails]
  Prompt: Find the professional email address, LinkedIn profile URL, Facebook page, YouTube channel, Twitter/X profile for Paper Trails (website: https://papertrails.podbean.com/). Check the website's contact and about pages. Return only verified, factual information with sources.


INFO:httpx:HTTP Request: POST https://api.perplexity.ai/chat/completions "HTTP/1.1 200 OK"


  Emails found:  ['contact@podbean.com']
  Website found: https://papertrails.podbean.com
  LinkedIn:      None


## Cell 5: Full Raw Responses

In [5]:
for name, data in perplexity_results.items():
    print(f"\n{'='*60}")
    print(f"Lead: {name}")
    print(f"Prompt: {data['prompt']}")
    print(f"\nResponse:\n{data['raw_response']}")
    print(f"\nParsed:")
    for k, v in data['parsed'].items():
        if v:
            print(f"  {k}: {v}")


Lead: A Mommy And A Mic
Prompt: Find the professional email address, LinkedIn profile URL, Facebook page, YouTube channel, Twitter/X profile for A Mommy And A Mic (website: http://www.amommyandamic.com/). Check the website's contact and about pages. Return only verified, factual information with sources.

Response:
No verified professional email address, LinkedIn profile URL, Facebook page, YouTube channel, or Twitter/X profile found for "A Mommy And A Mic" (website: http://www.amommyandamic.com/) in the provided search results or on its contact/about pages.

The search results do not include content from http://www.amommyandamic.com/ or matching contact details. The closest related result is for "Two Moms and a Mic" (a different entity), which lists briggz@comcast.net and 630-514-6972, but this is not verified for the queried website.[1] Other results [2]-[6] are unrelated (e.g., government sites, audio equipment, crime reporting).

Parsed:
  emails: ['briggz@comcast.net']
  website:

## Cell 6: Test a Single Custom Prompt

Quick cell to try a completely custom prompt on one lead.

In [6]:
# Edit these freely
CUSTOM_PROMPT = (
    "Find the professional email address of the host of the 'A Mommy And A Mic' podcast "
    "(website: http://www.amommyandamic.com/). "
    "Check the contact and about pages of the website. "
    "Return only verified, factual information with sources."
)

response = service.process_content(
    content="",
    prompt=CUSTOM_PROMPT,
    model=_PERPLEXITY_MODEL,
    temperature=_TEMPERATURE,
    max_tokens=_MAX_TOKENS,
)

raw = response.get("result", "")
parsed = parse_perplexity_response(raw)

print(f"Emails:  {parsed['emails']}")
print(f"Website: {parsed['website']}")
print(f"\nFull response:\n{raw}")

INFO:httpx:HTTP Request: POST https://api.perplexity.ai/chat/completions "HTTP/1.1 200 OK"


Emails:  ['podcast@myrockerbeez.com']
Website: http://www.amommyandamic.com

Full response:
**The host of the 'A Mommy And A Mic' podcast is associated with the professional email podcast@myrockerbeez.com.[6]**

This email is listed directly in the podcast's Apple Podcasts description for contacting the host, including for specific submissions like "Veggie Secrets," confirming its use for podcast-related professional communication.[6] No contact or about pages from http://www.amommyandamic.com/ appear in the provided search results, and no other verified emails match the exact podcast title across sources.[1][2][3][4][5][7]
